# Pol-ISAR Paper-Method OBB Replication (Isolated MMRotate Environment)

Reproduces detector families from the translated dataset paper using the
official MMRotate 1.x repository: O-RTMDet, O-FCOS, O-ATSS, S2A-Net and
O-Faster R-CNN. PSC is retained in the paper reference table but is not called
implemented because no official MMRotate PSC config exists.

The OpenMMLab stack is installed in an isolated Python 3.10 micromamba
environment. This avoids changing Colab's Pillow, requests, setuptools or
PyTorch packages, which caused the previous runtime failures.



In [ ]:
import os, re, json, shutil, subprocess, sys, textwrap, time
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

try:
    from google.colab import drive
    try: drive.mount('/content/drive')
    except Exception as exc: print('Drive mount message:', exc)
except Exception: pass

ROOTS=[Path('/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0'),
       Path('/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0/Pol_ISARSpaceTarget1.0'),
       Path('/content/Pol_ISARSpaceTarget1.0')]
ROOT=next((p for p in ROOTS if (p/'ImageSets/labelTxt').exists()),None)
if ROOT is None: raise FileNotFoundError('Pol-ISAR dataset root was not found.')
WORK=Path('/content/polisar_mmrotate_paper')
DATA=WORK/'dota_vv'
OUT=Path('/content/drive/MyDrive/drdo/polisar_paper_methods_mmrotate')
WORK.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
print('dataset:',ROOT,'output:',OUT)



## Build a duplicate-free DOTA dataset



In [ ]:
LABELS=ROOT/'ImageSets/labelTxt'; DAT=ROOT/'ImageSets/images'

def canonical_labels(split):
    keep={}
    for p in sorted((LABELS/split).glob('*.txt')):
        key=re.sub(r' \(\d+\)$','',p.stem)
        if key not in keep or p.stem==key: keep[key]=p
    paths=[keep[k] for k in sorted(keep)]
    expected={'train':1314,'valid':162,'test':162}[split]
    assert len(paths)==expected,(split,len(paths),expected)
    return paths

def vv_uint8(split,base):
    raw=np.fromfile(DAT/split/f'{base}_VV.dat',np.float32)
    assert raw.size==256*256*2,(base,raw.size)
    z=raw[0::2].reshape(256,256)+1j*raw[1::2].reshape(256,256)
    x=np.log1p(np.abs(z)); lo,hi=np.percentile(x,[1,99])
    return (255*np.clip((x-lo)/max(hi-lo,1e-6),0,1)).astype(np.uint8)

def build_dota(force=False):
    marker=DATA/'BUILD_COMPLETE.json'
    if marker.exists() and not force:
        audit=json.loads(marker.read_text())
        if audit.get('counts')=={'train':1314,'val':162,'test':162}:
            print('Using verified dataset:',DATA); return
    if DATA.exists(): shutil.rmtree(DATA)
    counts={}
    for src,dst in [('train','train'),('valid','val'),('test','test')]:
        (DATA/dst/'images').mkdir(parents=True); (DATA/dst/'annfiles').mkdir(parents=True)
        files=canonical_labels(src)
        for i,label in enumerate(files,1):
            base=re.sub(r' \(\d+\)$','',label.stem)
            img=vv_uint8(src,base)
            cv2.imwrite(str(DATA/dst/'images'/f'{base}.png'),np.repeat(img[...,None],3,axis=2))
            (DATA/dst/'annfiles'/f'{base}.txt').write_text(label.read_text())
            if i%100==0: print(dst,i,'/',len(files))
        counts[dst]=len(files)
    marker.write_text(json.dumps({'counts':counts,'source':'canonical labels only'},indent=2))
    print('built:',counts)

build_dota(force=False)



## Install MMRotate without modifying the notebook kernel

Run once per fresh Colab runtime. Installation is cached in `/content` only.



In [ ]:
MAMBA=Path('/content/bin/micromamba')
REPO=Path('/content/mmrotate-1x')
if not MAMBA.exists():
    subprocess.check_call("mkdir -p /content/bin && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba",shell=True)
if not REPO.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch','1.x','https://github.com/open-mmlab/mmrotate.git',str(REPO)])

def mmrun(*args):
    return [str(MAMBA),'run','-n','mmrotate',*map(str,args)]

ready=Path('/content/.mmrotate_paper_env_ready')
if not ready.exists():
    subprocess.check_call([str(MAMBA),'create','-y','-n','mmrotate','-c','conda-forge','python=3.10','pip'])
    subprocess.check_call(mmrun('pip','install','torch==2.0.1','torchvision==0.15.2','--index-url','https://download.pytorch.org/whl/cu118'))
    subprocess.check_call(mmrun('pip','install','openmim==0.3.9'))
    subprocess.check_call(mmrun('mim','install','mmengine==0.10.7','mmcv==2.0.1'))
    subprocess.check_call(mmrun('pip','install','mmdet==3.0.0','numpy<2','terminaltables','pycocotools'))
    subprocess.check_call(mmrun('pip','install','-e',str(REPO)))
    ready.write_text('ready')
subprocess.check_call(mmrun('python','-c','import torch,mmcv,mmdet,mmrotate; print(torch.__version__,torch.cuda.is_available(),mmcv.__version__,mmdet.__version__,mmrotate.__version__)'))



## Select methods and write controlled 100-epoch configs

Start with the translated paper's strongest O-RTMDet and a two-stage
O-Faster R-CNN comparator. Enable the remaining methods after these complete.



In [ ]:
CONFIGS={
 'O-RTMDet':'configs/rotated_rtmdet/rotated_rtmdet_m-3x-dota.py',
 'O-FCOS':'configs/rotated_fcos/rotated-fcos-le90_r50_fpn_1x_dota.py',
 'O-ATSS':'configs/rotated_atss/rotated-atss-le90_r50_fpn_1x_dota.py',
 'S2A-Net':'configs/s2anet/s2anet-le90_r50_fpn_1x_dota.py',
 'O-Faster-RCNN':'configs/rotated_faster_rcnn/rotated-faster-rcnn-le90_r50_fpn_1x_dota.py',
}
RUN_METHODS=['O-RTMDet','O-Faster-RCNN']
EPOCHS=100
def discover_classes():
    names=set()
    for label in canonical_labels('train'):
        for line in label.read_text().splitlines():
            parts=line.split()
            if len(parts)>=10: names.add(parts[8])
    return tuple(sorted(names))
CLASSES=discover_classes()
print('classes from training annotations:',CLASSES)
assert CLASSES, 'No classes found in training annotations.'

def model_override(name):
    if name == 'S2A-Net':
        return f"model = dict(bbox_head_init=dict(num_classes={len(CLASSES)}), bbox_head_refine=[dict(num_classes={len(CLASSES)})])"
    if name in {'O-RTMDet','O-FCOS','O-ATSS'}:
        return f"model = dict(bbox_head=dict(num_classes={len(CLASSES)}))"
    return f"model = dict(roi_head=dict(bbox_head=dict(num_classes={len(CLASSES)})))"

def write_cfg(name):
    base=(REPO/CONFIGS[name]).as_posix()
    path=WORK/f"{name.replace('-','_')}_polisar.py"
    text=f'''_base_ = r"{base}"
data_root = r"{DATA.as_posix()}/"
metainfo = dict(classes={CLASSES!r})
train_dataloader = dict(batch_size=4, num_workers=2, dataset=dict(data_root=data_root, ann_file='train/annfiles/', data_prefix=dict(img_path='train/images/'), metainfo=metainfo))
val_dataloader = dict(batch_size=1, num_workers=2, dataset=dict(data_root=data_root, ann_file='val/annfiles/', data_prefix=dict(img_path='val/images/'), metainfo=metainfo))
test_dataloader = dict(batch_size=1, num_workers=2, dataset=dict(data_root=data_root, ann_file='test/annfiles/', data_prefix=dict(img_path='test/images/'), metainfo=metainfo))
{model_override(name)}
optim_wrapper = dict(optimizer=dict(type='AdamW', lr=0.001, weight_decay=0.05))
train_cfg = dict(type='EpochBasedTrainLoop', max_epochs={EPOCHS}, val_interval=1)
default_hooks = dict(checkpoint=dict(type='CheckpointHook', interval=1, save_best='dota/mAP', rule='greater', max_keep_ckpts=3))
work_dir = r"{(OUT/'runs'/name).as_posix()}"
'''
    path.write_text(text); return path

custom={name:write_cfg(name) for name in RUN_METHODS}
display(pd.DataFrame([{'method':k,'config':str(v),'epochs':EPOCHS} for k,v in custom.items()]))



## Train, resume, and test

Results are written after each method. On disconnection rerun the notebook;
`--resume auto` continues from the latest Drive checkpoint.



In [ ]:
rows=[]
for name,cfg in custom.items():
    run=OUT/'runs'/name; run.mkdir(parents=True,exist_ok=True)
    cmd=mmrun('python',REPO/'tools/train.py',cfg,'--work-dir',run)
    if any(run.glob('*.pth')): cmd.append('--resume')
    started=time.time(); status='ok'; error=''
    try: subprocess.check_call(cmd)
    except Exception as exc: status='failed'; error=repr(exc)
    ckpts=sorted(run.glob('*.pth'),key=lambda p:p.stat().st_mtime)
    if status=='ok' and ckpts:
        try:
            test_proc=subprocess.run(mmrun('python',REPO/'tools/test.py',cfg,ckpts[-1],'--out',run/'test_predictions.pkl'),text=True,capture_output=True)
            (run/'test.log').write_text(test_proc.stdout+'\n'+test_proc.stderr)
            if test_proc.returncode: raise RuntimeError(f'test.py exited with {test_proc.returncode}')
        except Exception as exc: status='test_failed'; error=repr(exc)
    rows.append({'method':name,'status':status,'error':error,'elapsed_min':(time.time()-started)/60,'run_dir':str(run)})
    pd.DataFrame(rows).to_csv(OUT/'paper_methods_status.csv',index=False)
display(pd.DataFrame(rows))



## Extract test mAP and epoch curves

MMRotate's standard DOTA evaluator reports mAP. Precision, recall and F1 are
not invented here; add a confidence-threshold evaluator before reporting those
metrics for these methods.



In [ ]:
import matplotlib.pyplot as plt

def parse_test_map(path):
    if not path.exists(): return None
    text=path.read_text(errors='ignore')
    hits=re.findall(r"(?:dota/)?mAP[^0-9]+([0-9]*\.?[0-9]+)",text)
    return float(hits[-1]) if hits else None

metric_rows=[]
for name in RUN_METHODS:
    run=OUT/'runs'/name
    metric_rows.append({'method':name,'test_mAP':parse_test_map(run/'test.log')})
    scalar_files=list(run.rglob('scalars.json'))
    if not scalar_files: continue
    records=[]
    for line in scalar_files[-1].read_text(errors='ignore').splitlines():
        try: records.append(json.loads(line))
        except Exception: pass
    hist=pd.DataFrame(records)
    if hist.empty or 'epoch' not in hist: continue
    fig,ax=plt.subplots(1,2,figsize=(11,4))
    loss_cols=[c for c in hist.columns if 'loss' in c and pd.api.types.is_numeric_dtype(hist[c])]
    for col in loss_cols: ax[0].plot(hist['epoch'],hist[col],label=col)
    map_col=next((c for c in hist.columns if c.endswith('mAP')),None)
    if map_col: ax[1].plot(hist['epoch'],hist[map_col],label=map_col)
    ax[0].set(title=f'{name}: training/validation losses',xlabel='Epoch',ylabel='Loss')
    ax[1].set(title=f'{name}: validation mAP',xlabel='Epoch',ylabel='mAP',ylim=(0,1))
    for a in ax: a.grid(alpha=.25); a.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(OUT/f'{name}_epoch_curves.png',dpi=180); plt.show()
metrics_df=pd.DataFrame(metric_rows)
display(metrics_df); metrics_df.to_csv(OUT/'paper_methods_test_map.csv',index=False)



## Paper reference and reporting boundary



In [ ]:
paper=pd.DataFrame([
 ['O-RTMDet',.7911,.6992,.7282,.6828],['O-FCOS',.7242,.6928,.7041,.6657],
 ['O-Faster-RCNN',.6564,.6932,.6669,.6597],['O-ATSS',.7446,.6793,.7029,.6586],
 ['PSC',.6743,.6619,.6619,.6188],['S2A-Net',.5905,.5273,.5508,.4908]],
 columns=['method','paper_precision_mean','paper_recall_mean','paper_f1_mean','paper_mAP50_mean'])
display(paper); paper.to_csv(OUT/'translated_paper_reference_means.csv',index=False)
print('Do not place paper reference values in the column for our experimental results.')
